In [1]:
# Tutorial: Web Scraping y Enriquecimiento de Datos con PokeAPI
# Este script está diseñado para enseñar a futuros ingenieros de datos cómo combinar
# web scraping y APIs para crear un dataset estandarizado. Usaremos Pokémon Database
# (web scraping) y PokeAPI (API) para construir un dataset de Pokémon.

# Cada sección (chunk) está separada para claridad, con explicaciones en markdown
# y comentarios en el código. Sigue las instrucciones para ejecutar el código en un
# Jupyter Notebook.

# Dependencias necesarias:
# pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import logging
import time
from typing import List, Dict
import uuid

# Configuración inicial del logging para rastrear el proceso
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# --- CHUNK 1: Configuración y Verificación de Dependencias ---
# Objetivo: Configurar el entorno y verificar que las bibliotecas estén instaladas.
# Esto asegura que el código sea reproducible.

logger.info("Iniciando el pipeline de datos de Pokémon")

try:
    import requests
    import bs4
    import pandas
    logger.info("Todas las dependencias están instaladas correctamente")
except ImportError as e:
    logger.error(f"Error de importación: {e}. Instala las dependencias con 'pip install requests beautifulsoup4 pandas'")
    raise


2026-07-29 19:04:01,858 - INFO - Iniciando el pipeline de datos de Pokémon
2026-07-29 19:04:01,861 - INFO - Todas las dependencias están instaladas correctamente


In [2]:
# --- CHUNK 2: Web Scraping de Pokémon Database ---
# Objetivo: Extraer datos de la tabla en https://pokemondb.net/pokedex/all
# La tabla contiene ID, Nombre, Tipo(s), Total, HP, Attack, Defense, Sp. Atk, Sp. Def, Speed.
# Usamos BeautifulSoup para parsear el HTML y pandas para estructurar los datos.

def scrape_pokemondb(url: str) -> pd.DataFrame:
    """
    Realiza web scraping de la página de Pokémon Database y retorna un DataFrame.
    
    Args:
        url (str): URL de la página a scrapear.
    
    Returns:
        pd.DataFrame: DataFrame con los datos extraídos.
    """
    try:
        # Configuramos un encabezado para simular un navegador
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        logger.info(f"Realizando solicitud HTTP a {url}")
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()  # Lanza excepción si la solicitud falla
        
        # Parseamos el HTML
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Encontramos la tabla principal
        table = soup.find('table', {'id': 'pokedex'})
        if not table:
            raise ValueError("No se encontró la tabla con id='pokedex'")
        
        # Extraemos las filas
        rows = table.find_all('tr')[1:]  # Saltamos el encabezado
        data = []
        
        for row in rows:
            cols = row.find_all('td')
            if len(cols) < 10:
                logger.warning(f"Fila incompleta encontrada, saltando: {cols}")
                continue
            
            # Extraemos los datos de cada columna
            pokemon_id = cols[0].text.strip()
            name = cols[1].find('a', class_='ent-name').text.strip()
            types = [t.text.strip() for t in cols[2].find_all('a')]
            total = cols[3].text.strip()
            hp = cols[4].text.strip()
            attack = cols[5].text.strip()
            defense = cols[6].text.strip()
            sp_atk = cols[7].text.strip()
            sp_def = cols[8].text.strip()
            speed = cols[9].text.strip()
            
            data.append({
                'id': pokemon_id,
                'name': name,
                'types': types,
                'total': total,
                'hp': hp,
                'attack': attack,
                'defense': defense,
                'sp_atk': sp_atk,
                'sp_def': sp_def,
                'speed': speed
            })
        
        # Convertimos a DataFrame
        df = pd.DataFrame(data)
        logger.info(f"Scraping completado. Extraídos {len(df)} Pokémon")
        return df
    
    except requests.exceptions.RequestException as e:
        logger.error(f"Error en la solicitud HTTP: {e}")
        raise
    except Exception as e:
        logger.error(f"Error durante el scraping: {e}")
        raise

# Ejecutamos el scraping
url_pokemondb = "https://pokemondb.net/pokedex/all"
df_pokemondb = scrape_pokemondb(url_pokemondb)

# Mostramos las primeras filas para verificar
logger.info("Primeras 5 filas del DataFrame de Pokémon Database:")
print(df_pokemondb.head())




2026-07-29 19:04:10,408 - INFO - Realizando solicitud HTTP a https://pokemondb.net/pokedex/all
2026-07-29 19:04:12,713 - INFO - Scraping completado. Extraídos 1219 Pokémon
2026-07-29 19:04:12,715 - INFO - Primeras 5 filas del DataFrame de Pokémon Database:


     id        name            types total  hp attack defense sp_atk sp_def  \
0  0001   Bulbasaur  [Grass, Poison]   318  45     49      49     65     65   
1  0002     Ivysaur  [Grass, Poison]   405  60     62      63     80     80   
2  0003    Venusaur  [Grass, Poison]   525  80     82      83    100    100   
3  0003    Venusaur  [Grass, Poison]   625  80    100     123    122    120   
4  0004  Charmander           [Fire]   309  39     52      43     60     50   

  speed  
0    45  
1    60  
2    80  
3    80  
4    65  


In [4]:
# --- CHUNK 3: Conexión a la PokeAPI ---
# Objetivo: Obtener datos adicionales (peso, altura, habilidades) de la PokeAPI
# para cada Pokémon extraído. Usamos solicitudes individuales para cada Pokémon
# y manejamos los límites de tasa.

def fetch_pokemon_data(pokemon_name: str, max_retries: int = 3, delay: float = 1.0) -> Dict:
    """
    Obtiene datos de un Pokémon desde la PokeAPI.
    
    Args:
        pokemon_name (str): Nombre del Pokémon (en minúsculas).
        max_retries (int): Número máximo de reintentos en caso de error.
        delay (float): Tiempo de espera entre reintentos (segundos).
    
    Returns:
        Dict: Diccionario con peso, altura y habilidades.
    """
    url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_name.lower()}"
    for attempt in range(max_retries):
        try:
            logger.info(f"Solicitando datos para {pokemon_name} (intento {attempt + 1})")
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            
            data = response.json()
            abilities = [ability['ability']['name'] for ability in data['abilities']]
            return {
                'name': pokemon_name,
                'weight': data['weight'],
                'height': data['height'],
                'abilities': abilities
            }
        
        except requests.exceptions.HTTPError as e:
            if response.status_code == 404:
                logger.warning(f"Pokémon {pokemon_name} no encontrado en la API")
                return {'name': pokemon_name, 'weight': None, 'height': None, 'abilities': []}
            elif response.status_code == 429:
                logger.warning(f"Límite de tasa alcanzado, esperando {delay} segundos")
                time.sleep(delay)
            else:
                logger.error(f"Error HTTP para {pokemon_name}: {e}")
                time.sleep(delay)
        except Exception as e:
            logger.error(f"Error al obtener datos para {pokemon_name}: {e}")
            time.sleep(delay)
    
    logger.error(f"No se pudieron obtener datos para {pokemon_name} después de {max_retries} intentos")
    return {'name': pokemon_name, 'weight': None, 'height': None, 'abilities': []}

# Obtenemos datos de la API para cada Pokémon
api_data = []
for name in df_pokemondb['name']:
    pokemon_data = fetch_pokemon_data(name)
    api_data.append(pokemon_data)
    time.sleep(0.1)  # Breve pausa para respetar límites de tasa

# Convertimos a DataFrame
df_api = pd.DataFrame(api_data)
logger.info(f"Datos de la API obtenidos para {len(df_api)} Pokémon")

# Mostramos las primeras filas
logger.info("Primeras 5 filas del DataFrame de la PokeAPI:")
print(df_api.head())




2025-04-22 14:40:50,523 - INFO - Solicitando datos para Bulbasaur (intento 1)
2025-04-22 14:40:50,927 - INFO - Solicitando datos para Ivysaur (intento 1)
2025-04-22 14:40:51,201 - INFO - Solicitando datos para Venusaur (intento 1)
2025-04-22 14:40:51,584 - INFO - Solicitando datos para Venusaur (intento 1)
2025-04-22 14:40:51,875 - INFO - Solicitando datos para Charmander (intento 1)
2025-04-22 14:40:52,188 - INFO - Solicitando datos para Charmeleon (intento 1)
2025-04-22 14:40:52,489 - INFO - Solicitando datos para Charizard (intento 1)
2025-04-22 14:40:52,774 - INFO - Solicitando datos para Charizard (intento 1)
2025-04-22 14:40:53,060 - INFO - Solicitando datos para Charizard (intento 1)
2025-04-22 14:40:53,333 - INFO - Solicitando datos para Squirtle (intento 1)
2025-04-22 14:40:53,629 - INFO - Solicitando datos para Wartortle (intento 1)
2025-04-22 14:40:53,913 - INFO - Solicitando datos para Blastoise (intento 1)
2025-04-22 14:40:54,212 - INFO - Solicitando datos para Blastoise (

2025-04-22 14:41:21,222 - INFO - Solicitando datos para Slowpoke (intento 1)
2025-04-22 14:41:21,524 - INFO - Solicitando datos para Slowpoke (intento 1)
2025-04-22 14:41:21,807 - INFO - Solicitando datos para Slowbro (intento 1)
2025-04-22 14:41:22,109 - INFO - Solicitando datos para Slowbro (intento 1)
2025-04-22 14:41:22,393 - INFO - Solicitando datos para Slowbro (intento 1)
2025-04-22 14:41:22,662 - INFO - Solicitando datos para Magnemite (intento 1)
2025-04-22 14:41:22,959 - INFO - Solicitando datos para Magneton (intento 1)
2025-04-22 14:41:23,265 - INFO - Solicitando datos para Farfetch'd (intento 1)
2025-04-22 14:41:23,464 - WARNING - Pokémon Farfetch'd no encontrado en la API
2025-04-22 14:41:23,565 - INFO - Solicitando datos para Farfetch'd (intento 1)
2025-04-22 14:41:23,770 - WARNING - Pokémon Farfetch'd no encontrado en la API
2025-04-22 14:41:23,882 - INFO - Solicitando datos para Doduo (intento 1)
2025-04-22 14:41:24,197 - INFO - Solicitando datos para Dodrio (intento 1

2025-04-22 14:41:52,751 - INFO - Solicitando datos para Typhlosion (intento 1)
2025-04-22 14:41:53,049 - INFO - Solicitando datos para Typhlosion (intento 1)
2025-04-22 14:41:53,351 - INFO - Solicitando datos para Totodile (intento 1)
2025-04-22 14:41:53,634 - INFO - Solicitando datos para Croconaw (intento 1)
2025-04-22 14:41:53,937 - INFO - Solicitando datos para Feraligatr (intento 1)
2025-04-22 14:41:54,236 - INFO - Solicitando datos para Sentret (intento 1)
2025-04-22 14:41:54,522 - INFO - Solicitando datos para Furret (intento 1)
2025-04-22 14:41:54,823 - INFO - Solicitando datos para Hoothoot (intento 1)
2025-04-22 14:41:55,091 - INFO - Solicitando datos para Noctowl (intento 1)
2025-04-22 14:41:55,459 - INFO - Solicitando datos para Ledyba (intento 1)
2025-04-22 14:41:55,811 - INFO - Solicitando datos para Ledian (intento 1)
2025-04-22 14:41:56,077 - INFO - Solicitando datos para Spinarak (intento 1)
2025-04-22 14:41:56,393 - INFO - Solicitando datos para Ariados (intento 1)
20

2025-04-22 14:42:24,976 - INFO - Solicitando datos para Grovyle (intento 1)
2025-04-22 14:42:25,246 - INFO - Solicitando datos para Sceptile (intento 1)
2025-04-22 14:42:25,545 - INFO - Solicitando datos para Sceptile (intento 1)
2025-04-22 14:42:25,851 - INFO - Solicitando datos para Torchic (intento 1)
2025-04-22 14:42:26,135 - INFO - Solicitando datos para Combusken (intento 1)
2025-04-22 14:42:26,432 - INFO - Solicitando datos para Blaziken (intento 1)
2025-04-22 14:42:26,718 - INFO - Solicitando datos para Blaziken (intento 1)
2025-04-22 14:42:27,001 - INFO - Solicitando datos para Mudkip (intento 1)
2025-04-22 14:42:27,319 - INFO - Solicitando datos para Marshtomp (intento 1)
2025-04-22 14:42:27,634 - INFO - Solicitando datos para Swampert (intento 1)
2025-04-22 14:42:27,920 - INFO - Solicitando datos para Swampert (intento 1)
2025-04-22 14:42:28,220 - INFO - Solicitando datos para Poochyena (intento 1)
2025-04-22 14:42:28,508 - INFO - Solicitando datos para Mightyena (intento 1)

2025-04-22 14:42:58,893 - INFO - Solicitando datos para Anorith (intento 1)
2025-04-22 14:42:59,180 - INFO - Solicitando datos para Armaldo (intento 1)
2025-04-22 14:42:59,480 - INFO - Solicitando datos para Feebas (intento 1)
2025-04-22 14:42:59,783 - INFO - Solicitando datos para Milotic (intento 1)
2025-04-22 14:43:00,068 - INFO - Solicitando datos para Castform (intento 1)
2025-04-22 14:43:00,365 - INFO - Solicitando datos para Castform (intento 1)
2025-04-22 14:43:00,668 - INFO - Solicitando datos para Castform (intento 1)
2025-04-22 14:43:00,969 - INFO - Solicitando datos para Castform (intento 1)
2025-04-22 14:43:01,271 - INFO - Solicitando datos para Kecleon (intento 1)
2025-04-22 14:43:01,570 - INFO - Solicitando datos para Shuppet (intento 1)
2025-04-22 14:43:01,856 - INFO - Solicitando datos para Banette (intento 1)
2025-04-22 14:43:02,160 - INFO - Solicitando datos para Banette (intento 1)
2025-04-22 14:43:02,458 - INFO - Solicitando datos para Duskull (intento 1)
2025-04-2

2025-04-22 14:43:29,120 - INFO - Solicitando datos para Lopunny (intento 1)
2025-04-22 14:43:29,438 - INFO - Solicitando datos para Lopunny (intento 1)
2025-04-22 14:43:29,719 - INFO - Solicitando datos para Mismagius (intento 1)
2025-04-22 14:43:30,021 - INFO - Solicitando datos para Honchkrow (intento 1)
2025-04-22 14:43:30,322 - INFO - Solicitando datos para Glameow (intento 1)
2025-04-22 14:43:30,622 - INFO - Solicitando datos para Purugly (intento 1)
2025-04-22 14:43:30,908 - INFO - Solicitando datos para Chingling (intento 1)
2025-04-22 14:43:31,221 - INFO - Solicitando datos para Stunky (intento 1)
2025-04-22 14:43:31,512 - INFO - Solicitando datos para Skuntank (intento 1)
2025-04-22 14:43:31,845 - INFO - Solicitando datos para Bronzor (intento 1)
2025-04-22 14:43:32,144 - INFO - Solicitando datos para Bronzong (intento 1)
2025-04-22 14:43:32,429 - INFO - Solicitando datos para Bonsly (intento 1)
2025-04-22 14:43:32,729 - INFO - Solicitando datos para Mime Jr. (intento 1)
2025-

... (4 outputs left out)

2025-04-22 14:45:55,998 - INFO - Solicitando datos para Arrokuda (intento 1)
2025-04-22 14:45:56,298 - INFO - Solicitando datos para Barraskewda (intento 1)
2025-04-22 14:45:56,567 - INFO - Solicitando datos para Toxel (intento 1)
2025-04-22 14:45:56,865 - INFO - Solicitando datos para Toxtricity (intento 1)
2025-04-22 14:45:57,064 - WARNING - Pokémon Toxtricity no encontrado en la API
2025-04-22 14:45:57,165 - INFO - Solicitando datos para Toxtricity (intento 1)
2025-04-22 14:45:57,334 - WARNING - Pokémon Toxtricity no encontrado en la API
2025-04-22 14:45:57,435 - INFO - Solicitando datos para Sizzlipede (intento 1)
2025-04-22 14:45:57,717 - INFO - Solicitando datos para Centiskorch (intento 1)
2025-04-22 14:45:58,001 - INFO - Solicitando datos para Clobbopus (intento 1)
2025-04-22 14:45:58,303 - INFO - Solicitando datos para Grapploct (intento 1)
2025-04-22 14:45:58,619 - INFO - Solicitando datos para Sinistea (intento 1)
2025-04-22 14:45:58,925 - INFO - Solicitando datos para Polte

2025-04-22 14:46:22,789 - INFO - Solicitando datos para Pawmi (intento 1)
2025-04-22 14:46:23,059 - INFO - Solicitando datos para Pawmo (intento 1)
2025-04-22 14:46:23,340 - INFO - Solicitando datos para Pawmot (intento 1)
2025-04-22 14:46:23,642 - INFO - Solicitando datos para Tandemaus (intento 1)
2025-04-22 14:46:23,975 - INFO - Solicitando datos para Maushold (intento 1)
2025-04-22 14:46:24,180 - WARNING - Pokémon Maushold no encontrado en la API
2025-04-22 14:46:24,281 - INFO - Solicitando datos para Maushold (intento 1)
2025-04-22 14:46:24,463 - WARNING - Pokémon Maushold no encontrado en la API
2025-04-22 14:46:24,565 - INFO - Solicitando datos para Fidough (intento 1)
2025-04-22 14:46:24,849 - INFO - Solicitando datos para Dachsbun (intento 1)
2025-04-22 14:46:25,146 - INFO - Solicitando datos para Smoliv (intento 1)
2025-04-22 14:46:25,434 - INFO - Solicitando datos para Dolliv (intento 1)
2025-04-22 14:46:25,735 - INFO - Solicitando datos para Arboliva (intento 1)
2025-04-22 

2025-04-22 14:46:46,844 - WARNING - Pokémon Iron Moth no encontrado en la API
2025-04-22 14:46:46,958 - INFO - Solicitando datos para Iron Thorns (intento 1)
2025-04-22 14:46:47,143 - WARNING - Pokémon Iron Thorns no encontrado en la API
2025-04-22 14:46:47,258 - INFO - Solicitando datos para Frigibax (intento 1)
2025-04-22 14:46:47,541 - INFO - Solicitando datos para Arctibax (intento 1)
2025-04-22 14:46:47,827 - INFO - Solicitando datos para Baxcalibur (intento 1)
2025-04-22 14:46:48,112 - INFO - Solicitando datos para Gimmighoul (intento 1)
2025-04-22 14:46:48,410 - INFO - Solicitando datos para Gimmighoul (intento 1)
2025-04-22 14:46:48,732 - INFO - Solicitando datos para Gholdengo (intento 1)
2025-04-22 14:46:49,000 - INFO - Solicitando datos para Wo-Chien (intento 1)
2025-04-22 14:46:49,314 - INFO - Solicitando datos para Chien-Pao (intento 1)
2025-04-22 14:46:49,631 - INFO - Solicitando datos para Ting-Lu (intento 1)
2025-04-22 14:46:49,949 - INFO - Solicitando datos para Chi-Yu

         name  weight  height                abilities
0   Bulbasaur    69.0     7.0  [overgrow, chlorophyll]
1     Ivysaur   130.0    10.0  [overgrow, chlorophyll]
2    Venusaur  1000.0    20.0  [overgrow, chlorophyll]
3    Venusaur  1000.0    20.0  [overgrow, chlorophyll]
4  Charmander    85.0     6.0     [blaze, solar-power]


In [5]:
# --- CHUNK 4: Combinación de Datos ---
# Objetivo: Unir los datos de Pokémon Database y PokeAPI en un solo DataFrame,
# asegurando que las filas coincidan por nombre.

def merge_datasets(df_scrape: pd.DataFrame, df_api: pd.DataFrame) -> pd.DataFrame:
    """
    Combina los DataFrames de web scraping y API.
    
    Args:
        df_scrape (pd.DataFrame): DataFrame del scraping.
        df_api (pd.DataFrame): DataFrame de la API.
    
    Returns:
        pd.DataFrame: DataFrame combinado.
    """
    try:
        logger.info("Combinando datasets")
        # Unimos por nombre, usando un left join para mantener todos los Pokémon del scraping
        df_merged = pd.merge(
            df_scrape,
            df_api,
            on='name',
            how='left'
        )
        
        # Verificamos si hay duplicados
        if df_merged['name'].duplicated().any():
            logger.warning("Se encontraron nombres duplicados en el dataset combinado")
            df_merged = df_merged.drop_duplicates(subset='name', keep='first')
        
        logger.info(f"Datasets combinados. Total de filas: {len(df_merged)}")
        return df_merged
    
    except Exception as e:
        logger.error(f"Error al combinar datasets: {e}")
        raise

# Combinamos los datasets
df_final = merge_datasets(df_pokemondb, df_api)

# Mostramos las primeras filas
logger.info("Primeras 5 filas del DataFrame combinado:")
print(df_final.head())



2025-04-22 14:46:59,473 - INFO - Combinando datasets
2025-04-22 14:46:59,473 - WARNING - Se encontraron nombres duplicados en el dataset combinado
2025-04-22 14:46:59,473 - INFO - Datasets combinados. Total de filas: 1025
2025-04-22 14:46:59,473 - INFO - Primeras 5 filas del DataFrame combinado:


     id        name            types total  hp attack defense sp_atk sp_def  \
0  0001   Bulbasaur  [Grass, Poison]   318  45     49      49     65     65   
1  0002     Ivysaur  [Grass, Poison]   405  60     62      63     80     80   
2  0003    Venusaur  [Grass, Poison]   525  80     82      83    100    100   
6  0004  Charmander           [Fire]   309  39     52      43     60     50   
7  0005  Charmeleon           [Fire]   405  58     64      58     80     65   

  speed  weight  height                abilities  
0    45    69.0     7.0  [overgrow, chlorophyll]  
1    60   130.0    10.0  [overgrow, chlorophyll]  
2    80  1000.0    20.0  [overgrow, chlorophyll]  
6    65    85.0     6.0     [blaze, solar-power]  
7    80   190.0    11.0     [blaze, solar-power]  


In [6]:
# --- CHUNK 5: Limpieza y Estandarización ---
# Objetivo: Limpiar el dataset combinado, manejando valores nulos y estandarizando
# los tipos de datos para análisis posteriores.

def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia y estandariza el dataset.
    
    Args:
        df (pd.DataFrame): DataFrame a limpiar.
    
    Returns:
        pd.DataFrame: DataFrame limpio.
    """
    try:
        logger.info("Iniciando limpieza del dataset")
        df_clean = df.copy()
        
        # Convertimos columnas numéricas a enteros, manejando valores nulos
        numeric_cols = ['id', 'total', 'hp', 'attack', 'defense', 'sp_atk', 'sp_def', 'speed']
        for col in numeric_cols:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').astype('Int64')  # Int64 permite nulos
        
        # Manejo de valores nulos en peso y altura
        df_clean['weight'] = df_clean['weight'].fillna(0)  # 0 para peso desconocido
        df_clean['height'] = df_clean['height'].fillna(0)  # 0 para altura desconocida
        
        # Convertimos tipos a lista si no lo son
        df_clean['types'] = df_clean['types'].apply(lambda x: x if isinstance(x, list) else [])
        df_clean['abilities'] = df_clean['abilities'].apply(lambda x: x if isinstance(x, list) else [])
        
        # Eliminamos filas sin ID o nombre
        df_clean = df_clean.dropna(subset=['id', 'name'])
        
        logger.info(f"Limpieza completada. Total de filas: {len(df_clean)}")
        return df_clean
    
    except Exception as e:
        logger.error(f"Error durante la limpieza: {e}")
        raise

# Limpiamos el dataset
df_clean = clean_dataset(df_final)

# Mostramos las primeras filas
logger.info("Primeras 5 filas del DataFrame limpio:")
print(df_clean.head())


2025-04-22 14:46:59,516 - INFO - Iniciando limpieza del dataset
2025-04-22 14:46:59,525 - INFO - Limpieza completada. Total de filas: 1025
2025-04-22 14:46:59,525 - INFO - Primeras 5 filas del DataFrame limpio:


   id        name            types  total  hp  attack  defense  sp_atk  \
0   1   Bulbasaur  [Grass, Poison]    318  45      49       49      65   
1   2     Ivysaur  [Grass, Poison]    405  60      62       63      80   
2   3    Venusaur  [Grass, Poison]    525  80      82       83     100   
6   4  Charmander           [Fire]    309  39      52       43      60   
7   5  Charmeleon           [Fire]    405  58      64       58      80   

   sp_def  speed  weight  height                abilities  
0      65     45    69.0     7.0  [overgrow, chlorophyll]  
1      80     60   130.0    10.0  [overgrow, chlorophyll]  
2     100     80  1000.0    20.0  [overgrow, chlorophyll]  
6      50     65    85.0     6.0     [blaze, solar-power]  
7      65     80   190.0    11.0     [blaze, solar-power]  


In [7]:

# --- CHUNK 6: Guardado del Dataset ---
# Objetivo: Guardar el dataset final en un archivo CSV para reusabilidad.

def save_dataset(df: pd.DataFrame, filename: str = f"pokemon_dataset_{uuid.uuid4().hex}.csv"):
    """
    Guarda el dataset en un archivo CSV.
    
    Args:
        df (pd.DataFrame): DataFrame a guardar.
        filename (str): Nombre del archivo CSV.
    """
    try:
        logger.info(f"Guardando dataset en {filename}")
        df.to_csv(filename, index=False)
        logger.info("Dataset guardado exitosamente")
    except Exception as e:
        logger.error(f"Error al guardar el dataset: {e}")
        raise

# Guardamos el dataset
save_dataset(df_clean)

# --- CHUNK 7: Resumen y Próximos Pasos ---
# Objetivo: Resumir el proceso y sugerir cómo usar el dataset para análisis futuros.

logger.info("Pipeline completado exitosamente!")
logger.info("Resumen:")
logger.info(f" - Pokémon extraídos del scraping: {len(df_pokemondb)}")
logger.info(f" - Pokémon con datos de la API: {len(df_api)}")
logger.info(f" - Filas en el dataset final: {len(df_clean)}")

# --- CHUNK 6: Guardado del Dataset ---
# Objetivo: Guardar el dataset final en un archivo CSV para reusabilidad.

def save_dataset(df: pd.DataFrame, filename: str = f"pokemon_dataset_{uuid.uuid4().hex}.csv"):
    """
    Guarda el dataset en un archivo CSV.
    
    Args:
        df (pd.DataFrame): DataFrame a guardar.
        filename (str): Nombre del archivo CSV.
    """
    try:
        logger.info(f"Guardando dataset en {filename}")
        df.to_csv(filename, index=False)
        logger.info("Dataset guardado exitosamente")
    except Exception as e:
        logger.error(f"Error al guardar el dataset: {e}")
        raise

# Guardamos el dataset
save_dataset(df_clean)



2025-04-22 14:46:59,540 - INFO - Guardando dataset en pokemon_dataset_28264340f37046279736c351facc0b11.csv
2025-04-22 14:46:59,553 - INFO - Dataset guardado exitosamente
2025-04-22 14:46:59,554 - INFO - Pipeline completado exitosamente!
2025-04-22 14:46:59,555 - INFO - Resumen:
2025-04-22 14:46:59,555 - INFO -  - Pokémon extraídos del scraping: 1215
2025-04-22 14:46:59,556 - INFO -  - Pokémon con datos de la API: 1215
2025-04-22 14:46:59,557 - INFO -  - Filas en el dataset final: 1025
2025-04-22 14:46:59,557 - INFO - Guardando dataset en pokemon_dataset_30830ecf2d7a4e6387daf6b91f99d91e.csv
2025-04-22 14:46:59,557 - INFO - Dataset guardado exitosamente


In [10]:
print(df_clean)

        id          name               types  total   hp  attack  defense  \
0        1     Bulbasaur     [Grass, Poison]    318   45      49       49   
1        2       Ivysaur     [Grass, Poison]    405   60      62       63   
2        3      Venusaur     [Grass, Poison]    525   80      82       83   
6        4    Charmander              [Fire]    309   39      52       43   
7        5    Charmeleon              [Fire]    405   58      64       58   
...    ...           ...                 ...    ...  ...     ...      ...   
1688  1021   Raging Bolt  [Electric, Dragon]    590  125      73       91   
1689  1022  Iron Boulder     [Rock, Psychic]    590   90     120       80   
1690  1023    Iron Crown    [Steel, Psychic]    590   90      72      100   
1691  1024     Terapagos            [Normal]    450   90      65       85   
1700  1025     Pecharunt     [Poison, Ghost]    600   88      88      160   

      sp_atk  sp_def  speed  weight  height                abilities  
0   